# Lab Day 10 — Qwen3-VL-4B Road Damage Grounding
Nguyễn Tuấn Thành — 2A202602640.

Notebook gọi cùng mã trong `lab/` với các lệnh terminal để tránh có hai implementation khác nhau.
Môi trường chạy chính: máy trường, tài khoản research02, GPU vật lý 1 RTX 6000 Ada.
Kết quả thật được ghi ra `report/evidence/`; notebook không chứa số liệu mẫu hoặc khóa truy cập.
Xem `PLAN.md` và `CHECKLIST.md` để biết checkpoint đã kiểm chứng. Mã GPU cần smoke test trên server trước chạy dài.


## 0. Môi trường (GPU trường RTX 6000 Ada)
Mở notebook bằng kernel thuộc `.venv` của dự án; cài `requirements-gpu.txt` trong venv.
Khởi động kernel với `CUDA_VISIBLE_DEVICES=1` trước khi import torch.
Nếu dùng terminal: `bash run_gpu.sh <checkpoint>`; chạy trong tmux để không mất job khi SSH ngắt.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

assert Path('lab/data.py').exists(), 'Mở notebook từ thư mục gốc dự án.'
if 'torch' in sys.modules and os.environ.get('CUDA_VISIBLE_DEVICES') != '1':
    raise RuntimeError('Restart kernel với CUDA_VISIBLE_DEVICES=1 trước khi import torch.')
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def run_module(module, *arguments):
    subprocess.run([sys.executable, '-m', module, *arguments], check=True)

subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,utilization.gpu', '--format=csv'], check=True)


In [ ]:
# Cấp credential bằng môi trường hoặc nhập ẩn; không lưu khóa vào notebook.
import getpass
if not os.environ.get('KAGGLE_USERNAME'):
    os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ').strip()
if not os.environ.get('KAGGLE_KEY'):
    os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle API key: ')


## 1. Dữ liệu: tải và chia split
Checkpoint `prepare` chạy pytest gốc trước tải, chạy prepare_data.py không sửa, audit split và test dữ liệu thật.
Chỉ chuyển bước khi số ảnh/box khớp bảng đề và không có session ở nhiều split; commit checkpoint sau khi kiểm chứng.


In [ ]:
run_module('lab.run', '--phase', 'prepare')


In [ ]:
from IPython.display import display, Image as DisplayImage
audit = json.loads(Path('report/runs/data_audit.json').read_text())
print(json.dumps(audit, indent=2, ensure_ascii=False))
for figure in sorted(Path('report/figures').glob('labels_*.png'))[:6]:
    display(DisplayImage(filename=str(figure)))


## 2. Nhãn: YOLO → JSON thang 0–1000
Hàm chuyển nhãn nằm trong lab/data.py: round/clamp, bảo đảm box còn diện tích tại biên, sắp (y1,x1).
Các kiểm thử CPU bao gồm round-trip về pixel, ảnh rỗng và box sát biên.


In [ ]:
from lab.data import yolo_to_box, objects_to_answer, PROMPT
from scripts.labcommon import read_jsonl
train_records = read_jsonl('data/splits/train.jsonl')
print(PROMPT)
print(objects_to_answer(train_records[0]['objects']))
run_module('pytest', '-q', 'student_tests/test_grounding.py')


## 3. E0: zero-shot
Chạy model gốc NF4 trên đủ 311 ảnh test. Visual/lm_head không 4-bit; giữ nguyên raw output.
Model revision được ghim một lần trong report/runs/model_revision.json. Chấm bằng scorer gốc trong cùng checkpoint.


In [ ]:
run_module('lab.run', '--phase', 'e0')


In [ ]:
e0 = json.loads(Path('report/evidence/e0.json').read_text())
print(json.dumps(e0, indent=2, ensure_ascii=False))
for record in read_jsonl('work/e0.jsonl')[:5]:
    print(record['image_id'], record['raw'])


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Prefix alignment được assert. Chỉ answer/EOS có loss; prompt/ảnh/padding đều -100.
Collate pad văn bản, nối tensor ảnh; giữ mm_token_type_ids do processor trả về.
Checkpoint encoding chạy forward/backward thật cho E1/E2/E3, kiểm tra 220 vision tokens và gradient đúng khối.


In [ ]:
from lab.data import encode_example, GroundingCollator
run_module('lab.run', '--phase', 'encoding')
encoding = json.loads(Path('report/runs/preflight_e1.json').read_text())
print('Image grid:', encoding['image_grid_thw'])
print('Vision tokens:', encoding['vision_tokens'])
print('Decoded supervised tokens:', encoding['decoded_answer_and_eos'])


## 5. Setup LoRA / đóng băng cho E1, E2, E3
E1/E2: 252 LLM modules; E3: thêm 96 ViT modules, gồm attention.proj, tổng 348.
E2/E3 mở cả merger và deepstack_merger_list ở float32. Không lượng tử hóa projector.


In [ ]:
from lab.modeling import apply_setup, is_projector
for name in ('e1', 'e2', 'e3'):
    checks = json.loads(Path(f'report/runs/preflight_{name}.json').read_text())
    stats = {key: value for key, value in checks['setup_stats'].items() if key != 'matched_module_names'}
    print(name, json.dumps(stats, indent=2))


## 6. Vòng huấn luyện
Mỗi checkpoint e1/e2/e3 khởi tạo từ cùng model gốc; 300 optimizer updates, AdamW, warmup/cosine, fp16 scaler.
Lưu checkpoint mỗi 100 bước: adapter, projector, optimizer/scheduler/scaler, RNG và thứ tự lấy dữ liệu.
Ghi số tham số/peak VRAM/phút thật. Chạy từng cell theo thứ tự, commit sau checkpoint có evidence.


In [ ]:
run_module('lab.run', '--phase', 'e1')


In [ ]:
run_module('lab.run', '--phase', 'e2')


In [ ]:
run_module('lab.run', '--phase', 'e3')


## 7. Suy luận và chấm E1–E3
Các checkpoint trên đã fresh reload rồi chạy val và test. Nạp projector kiểm tra key/shape/dtype/checksum.
Xem bằng chứng gốc; không dùng test để chọn hyperparameter/checkpoint.


In [ ]:
for name in ('e1', 'e2', 'e3'):
    evidence = json.loads(Path(f'report/evidence/{name}.json').read_text())
    print(name, 'F1:', evidence['micro']['f1'], 'CI95:', evidence['f1_ci95'],
          'valid:', evidence['parse_valid_rate'], 'meta:', evidence['meta'])


## 8. E4 (ablation rank) và E5 (tuỳ chọn, GRPO)
Setup nền E4 chọn bằng đầy đủ val. Hai run rank 4/64, alpha=2r; mọi yếu tố còn lại bằng setup nền.
Hàm reward E5 và test CPU đã có trong lab/reward.py; trainer GRPO cần xác minh API bản TRL trên server trước chạy bonus.


In [ ]:
run_module('lab.run', '--phase', 'e4_rank4')


In [ ]:
run_module('lab.run', '--phase', 'e4_rank64')


In [ ]:
run_module('pytest', '-q', 'student_tests/test_reward.py')


## 9. Phân tích và nghiệm thu
Chỉ chạy khi đủ evidence E0–E4. Finalize sinh ảnh, biểu đồ, báo cáo bằng số thật rồi chạy checker.
Trước đó hoàn tất các commit checkpoint để checker kiểm tra Git. Lưu nguyên test template của đề và công bố mâu thuẫn placeholder khi báo cáo đã điền.


In [ ]:
run_module('lab.finalize')
